# KNN
### 목표
1. KNN에서 K값이 성능에 어떤 영향을 주는지 확인
2. 표준화 전/후 성능 차이 확인
3. train 정확도와 test 정확도를 비교해서 과적합/과소적합 감 잡기

---

# 1. 데이터 불러오기

In [40]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("클래스 이름:", cancer.target_names)

X_train shape: (455, 30)
X_test shape: (114, 30)
클래스 이름: ['malignant' 'benign']


---

# 2. K값 하나로 먼저 확인

일단 K=3 으로 표준화 없음 / 표준화 적용 비교

In [41]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 표준화 적용 X
model_no_scaled = KNeighborsClassifier(n_neighbors=3)

model_no_scaled.fit(X_train, y_train)

train_no_scaled = model_no_scaled.score(X_train, y_train)
test_no_scaled = model_no_scaled.score(X_test, y_test)

# 표준화 적용
model_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=3))
])

model_scaled.fit(X_train, y_train)

train_scaled = model_scaled.score(X_train, y_train)
test_scaled = model_scaled.score(X_test, y_test)

print("표준화 없음")
print("Train 정확도:", train_no_scaled)
print("Test 정확도:", test_no_scaled)

print()

print("표준화 적용")
print("Train 정확도:", train_scaled)
print("Test 정확도:", test_scaled)

표준화 없음
Train 정확도: 0.9538461538461539
Test 정확도: 0.9298245614035088

표준화 적용
Train 정확도: 0.978021978021978
Test 정확도: 0.9824561403508771


위에서 표준화 적용 후 Test 정확도가 올라갔고
- ### KNN은 feature scale의 영향을 많이 받는다

라고 해석하면 된다.

---

# 3. K값 여러 개 비교하기

K = 1, 3, 5, ..., 29 까지 바꿔가면서 성능 비교

In [42]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

results = []

for k in range(1, 30, 2):
    # 표준화 안 함
    model_no_scaled = KNeighborsClassifier(n_neighbors=k)
    model_no_scaled.fit(X_train, y_train)

    train_no_scaled = model_no_scaled.score(X_train, y_train)
    test_no_scaled = model_no_scaled.score(X_test, y_test)

    # 표준화 적용
    model_scaled = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

    model_scaled.fit(X_train, y_train)

    train_scaled = model_scaled.score(X_train, y_train)
    test_scaled = model_scaled.score(X_test, y_test)

    results.append({
        "K": k,
        "Train_표준화없음": train_no_scaled,
        "Test_표준화없음": test_no_scaled,
        "Train_표준화적용": train_scaled,
        "Test_표준화적용": test_scaled
    })

result_df = pd.DataFrame(results)
result_df

,K,Train_표준화없음,Test_표준화없음,Train_표준화적용,Test_표준화적용
0,1,1.000000,0.921053,1.000000,0.938596
1,3,0.953846,0.929825,0.978022,0.982456
2,5,0.947253,0.912281,0.973626,0.956140
3,7,0.947253,0.929825,0.975824,0.973684
4,9,0.945055,0.938596,0.973626,0.973684
5,11,0.940659,0.938596,0.971429,0.973684
6,13,0.940659,0.947368,0.971429,0.973684
7,15,0.942857,0.929825,0.969231,0.973684
8,17,0.936264,0.938596,0.962637,0.982456
9,19,0.931868,0.938596,0.967033,0.973684


---

# 4. Test 정확도 높은 순서로 보기

표준화를 적용한 KNN 기준으로 가장 좋은 K값을 확인해보면

In [43]:
result_df.sort_values("Test_표준화적용", ascending=False)

,K,Train_표준화없음,Test_표준화없음,Train_표준화적용,Test_표준화적용
1,3,0.953846,0.929825,0.978022,0.982456
8,17,0.936264,0.938596,0.962637,0.982456
4,9,0.945055,0.938596,0.973626,0.973684
5,11,0.940659,0.938596,0.971429,0.973684
6,13,0.940659,0.947368,0.971429,0.973684
7,15,0.942857,0.929825,0.969231,0.973684
3,7,0.947253,0.929825,0.975824,0.973684
9,19,0.931868,0.938596,0.967033,0.973684
10,21,0.934066,0.938596,0.964835,0.964912
12,25,0.934066,0.938596,0.964835,0.964912
